In [2]:
import pandas as pd
import pickle
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import FeatureUnion
from geoai.utils_ds.preprocessing_ops import PreProcessingOperations
from sklearn.base import BaseEstimator, TransformerMixin
from geoai.utils_ml.model_ops import ModelOperations
from geoai.utils_geo.raster_ops import RasterOperations

preprocess_ops = PreProcessingOperations()
model_ops = ModelOperations()
raster_ops = RasterOperations()


# We will create a pipeline using the following steps:

#### 1. Load the data containing only the bands.

#### 2. Compute indices

#### 3. Bin and Categorize NDVI

#### 4. Pipeline 1:

- Apply Log transformation to numerical features

- Do a Polynomial transformation

- One hot encode NDVI_binary

- Ordinal encode NDVI_category

#### 5. Pipeline 2:

- Pipeline 1

- Scale to 0-1

- Apply LDA

#### 6. Pipeline 3:

- Pipeline 1

- Scale to 0-1

- Apply PCA

#### 7. Pipeline 4:

- Combine Pipeline 1, Pipeline 2, Pipeline 3

- Select Features

- Train a logisitic regression model

#### Step 1

In [3]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train_encoded.csv")
y_test = pd.read_csv("csv_files/y_test_encoded.csv")

#### Step 2 and 3

In [4]:
X_train = raster_ops.indices_binary_category(X_train) 
X_test = raster_ops.indices_binary_category(X_test)
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_categorized,NDVI_binary
0,350.0,542.0000,323.0,3277.0000,1975.3334,0.820556,-0.247826,0.002545,high_veg,veg
1,390.0,555.0000,380.0,3016.6667,1991.0000,0.776251,-0.204819,0.002227,high_veg,veg
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000,0.044101,0.098425,0.000127,low_veg,non_veg
3,363.2,546.5000,395.0,3244.5000,2052.0000,0.782937,-0.225149,0.002438,high_veg,veg
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000,0.031066,0.123170,0.000098,low_veg,non_veg


#### Step 4: Pipeline 1

In [10]:
# Define the column lists
numerical_columns = X_train.select_dtypes(include=["float64"]).columns.tolist()
one_hot_encoder_columns = ["NDVI_binary"]
ordinal_encoder_columns = ["NDVI_categorized"]
categories = [["low_veg", "medium_veg", "high_veg"]]

log_transformer = FunctionTransformer(np.log1p)
poly_transformer = PolynomialFeatures(degree=2, include_bias=False)
onehot_encoder = OneHotEncoder(dtype=int, sparse_output=False)
ordinal_encoder = OrdinalEncoder(categories=categories, dtype=int)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical_transformer",
            Pipeline(
                steps=[
                    ("log_transformer", log_transformer),
                    ("poly_transformer", poly_transformer),
                ]
            ),
            numerical_columns,
        ),
        ("onehot_encoder", onehot_encoder, one_hot_encoder_columns),
        ("ordinal_encoder", ordinal_encoder, ordinal_encoder_columns),
    ]
)

# Pipeline 1: Preprocessing
pipeline_1 = Pipeline(steps=[("preprocessor", preprocessor)])
pipeline_1

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical_transformer',
                                                  Pipeline(steps=[('log_transformer',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('poly_transformer',
                                                                   PolynomialFeatures(include_bias=False))]),
                                                  ['BLUE', 'GREEN', 'RED',
                                                   'NIR', 'SWIR', 'NDVI',
                                                   'NDBI', 'REI']),
                                                 ('onehot_encoder',
                                                  OneHotEncoder(dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['NDVI_binary']),
                                                 ('ordinal_encoder',
                                                  OrdinalEncoder(categories=[['low_veg',
                                                                              'medium_veg',
                                                                              'high_veg']],
                                                                 dtype=<class 'int'>),
                                                  ['NDVI_categorized'])]))])

#### Step 5: Pipeline 2

In [12]:
# Pipeline 2: Pipeline 1 + LDA
lda_transformer = LinearDiscriminantAnalysis(n_components=3) # Instantiate LDA
min_max_scaler = MinMaxScaler() # Instantiate MinMaxScaler

pipeline_2 = Pipeline(
    steps=[
        ("pipeline_1", pipeline_1),
        ("min_max_scaler", min_max_scaler),
        ("lda_transformer", lda_transformer),
    ]
)
pipeline_2

Pipeline(steps=[('pipeline_1',
                 Pipeline(steps=[('preprocessor',
                                  ColumnTransformer(transformers=[('numerical_transformer',
                                                                   Pipeline(steps=[('log_transformer',
                                                                                    FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                                   ('poly_transformer',
                                                                                    PolynomialFeatures(include_bias=False))]),
                                                                   ['BLUE',
                                                                    'GREEN',
                                                                    'RED',
                                                                    'NIR',
                                                                    'SWIR',
                                                                    'NDVI',
                                                                    'NDBI',
                                                                    'REI']),
                                                                  ('onehot_encoder',
                                                                   OneHotEncoder(dtype=<class 'int'>,
                                                                                 sparse_output=False),
                                                                   ['NDVI_binary']),
                                                                  ('ordinal_encoder',
                                                                   OrdinalEncoder(categories=[['low_veg',
                                                                                               'medium_veg',
                                                                                               'high_veg']],
                                                                                  dtype=<class 'int'>),
                                                                   ['NDVI_categorized'])]))])),
                ('min_max_scaler', MinMaxScaler()),
                ('lda_transformer',
                 LinearDiscriminantAnalysis(n_components=3))])

#### Step 6: Pipeline 3

In [15]:
# Pipeline 3: Pipeline 1 + PCA
pca_transformer = PCA(n_components=7)

pipeline_3 = Pipeline(
    steps=[
        ("pipeline_1", pipeline_1),
        ("min_max_scaler", min_max_scaler),
        ("pca_transformer", pca_transformer),
    ]
)
pipeline_3

Pipeline(steps=[('pipeline_1',
                 Pipeline(steps=[('preprocessor',
                                  ColumnTransformer(transformers=[('numerical_transformer',
                                                                   Pipeline(steps=[('log_transformer',
                                                                                    FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                                   ('poly_transformer',
                                                                                    PolynomialFeatures(include_bias=False))]),
                                                                   ['BLUE',
                                                                    'GREEN',
                                                                    'RED',
                                                                    'NIR',
                                                                    'SWIR',
                                                                    'NDVI',
                                                                    'NDBI',
                                                                    'REI']),
                                                                  ('onehot_encoder',
                                                                   OneHotEncoder(dtype=<class 'int'>,
                                                                                 sparse_output=False),
                                                                   ['NDVI_binary']),
                                                                  ('ordinal_encoder',
                                                                   OrdinalEncoder(categories=[['low_veg',
                                                                                               'medium_veg',
                                                                                               'high_veg']],
                                                                                  dtype=<class 'int'>),
                                                                   ['NDVI_categorized'])]))])),
                ('min_max_scaler', MinMaxScaler()),
                ('pca_transformer', PCA(n_components=7))])

#### Step 7: Pipeline 4

In [16]:
# Combine all pipelines
combined_features = FeatureUnion(
    [("pipeline_1", pipeline_1), ("pipeline_2", pipeline_2), ("pipeline_3", pipeline_3)]
)

# Indices of important features which was selected from the feature selection
# process in the past notebook
important_feature_indices = [56, 34, 4, 49, 55, 35, 52, 40, 5, 47, 20, 30, 54, 50, 48, 3, 42,]

# Custom transformer to select important features by indices
class ImportantFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, feature_indices):
        self.feature_indices = feature_indices

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[:, self.feature_indices]

In [17]:
# Finally, we use the combined featues and the important feature selector to
# train the logistic regression model
lr = LogisticRegression(solver="lbfgs", max_iter=10000, random_state=1)
pipeline_4 = Pipeline(
    steps=[
        ("combined_features", combined_features),
        ("feature_selector", ImportantFeatureSelector(important_feature_indices)),
        ("lr", lr),
    ]
)
pipeline_4

Pipeline(steps=[('combined_features',
                 FeatureUnion(transformer_list=[('pipeline_1',
                                                 Pipeline(steps=[('preprocessor',
                                                                  ColumnTransformer(transformers=[('numerical_transformer',
                                                                                                   Pipeline(steps=[('log_transformer',
                                                                                                                    FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                                                                   ('poly_transformer',
                                                                                                                    PolynomialFeatures(include_bias=False))]),
                                                                                                   ['BLUE',
                                                                                                    'GREEN',
                                                                                                    'RED',
                                                                                                    'NIR',
                                                                                                    'SWIR',
                                                                                                    'NDVI...
                                                                                                                    OrdinalEncoder(categories=[['low_veg',
                                                                                                                                                'medium_veg',
                                                                                                                                                'high_veg']],
                                                                                                                                   dtype=<class 'int'>),
                                                                                                                    ['NDVI_categorized'])]))])),
                                                                 ('min_max_scaler',
                                                                  MinMaxScaler()),
                                                                 ('pca_transformer',
                                                                  PCA(n_components=7))]))])),
                ('feature_selector',
                 ImportantFeatureSelector(feature_indices=[56, 34, 4, 49, 55,
                                                           35, 52, 40, 5, 47,
                                                           20, 30, 54, 50, 48,
                                                           3, 42])),
                ('lr', LogisticRegression(max_iter=10000, random_state=1))])

In [18]:
# save the model using pickle
# merge the train and test datasets
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

# train the model using the best hyperparameters and the whole dataset
pipeline_4.fit(X_all, y_all)
with open('trained_models/selected_features_pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline_4, file)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


END